<a href="https://colab.research.google.com/github/trainocate-japan/developing-agentic-ai-with-langchain/blob/main/chap03/hands-on/chap03_handson_3A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ハンズオン 3-A: create_agent で最初のエージェント

**研修コース「LangChain による Agentic AI 開発実践」/ 第3章「エージェント開発の基本」**

このハンズオンは、講師の解説を聞きながら**作成済みのセルを上から順に一緒に実行する**形式です。
受講者がコードを書く場面はありません (コードを書くのは演習 3-B です)。
まずは「動かして観察する」ことに集中してください。

題材は公式 quickstart 準拠の**天気エージェント** (`get_weather`) です。
中立な教材で部品を一巡し、ヘルプデスクへの応用は演習 3-B で行います。

## この Notebook で学ぶこと

第2章では、ツール定義の JSON Schema を手書きし、`tool_calls` を `json.loads` でパースし、
`tool_call_id` を取り違えないよう履歴を積む——という while ループを**自分の手で**書きました。
本章では、その一連の流れが LangChain の `create_agent` でわずか数行になることを、動かして確かめます。

1. **モデルの初期化 (3-1)** — `init_chat_model("openai:gpt-5.4")` でモデルを初期化し、`invoke` の戻り値が
   `AIMessage` (文字列ではない) であることを確認。`temperature` 0 / 1 で出力の揺れを観察する
2. **Messages (3-2)** — `SystemMessage` / `HumanMessage` / `AIMessage` で会話履歴を組み立てて `invoke`。
   `usage_metadata` と `content_blocks` を観察し、dict 形式 (`{"role": ...}`) も等価であることを確認する
3. **create_agent (3-3)** — `get_weather` ツール + `create_agent` で天気エージェントを構築。
   `result["messages"]` の軌跡 (Human → AI(tool_calls) → Tool → AI) をダンプし、第2章の手動ループと見比べる。
   戻り値が `CompiledStateGraph` であること、そして「さっきの都市」を覚えていない**ステートレス性**を体験する
4. **@tool (3-4)** — 2 つ目のツールを `@tool` で追加し、docstring の品質がツール選択に与える影響を比較する

## 前提条件

- **Google アカウント**を持っていること
- このファイルを **Google Colab** で開いていること
- Colab の **[シークレット]** に `OPENAI_API_KEY` を登録済みであること
  (第1章の演習 1-1 で登録済みのはずです。未登録でも、後述の「0-2. API キーのセットアップ」で登録できます)
- インターネット接続 (API を呼び出します)

## 所要時間

約 33 分 (3-1 〜 3-4 を、講師の解説を含めて一気通貫で実行)

---
> **モデル名について**: 本教材ではモデル名を変数 `MODEL` に集約しています。教材中の例は `MODEL = "openai:gpt-5.4"` ですが、
> **研修実施時には講師が指定する最新モデル名に差し替えてください**。1 箇所 (準備セル) を直すだけで全セルに反映されます。


## 0. セットアップ

### 0-1. 依存パッケージのインストール

LangChain v1 本体 (`langchain`) と OpenAI 統合 (`langchain-openai`) をインストールします。
Colab には未インストール、または古いバージョンが入っていることがあるため、`-U` で最新へ更新します。

> 研修実施時は再現性のため、バージョンをピン留めすることを推奨します
> (本コースの基盤は **langchain 1.3.x / langchain-openai 1.3.x** です)。
> 例: `!pip install -U "langchain==1.3.7" "langchain-openai==1.3.0"`


In [ ]:
# LangChain v1 本体と OpenAI 統合を最新版へインストール/更新
# 研修実施時はバージョンをピン留め推奨 (langchain 1.3.x / langchain-openai 1.3.x)
!pip install -U langchain langchain-openai

### 0-2. API キーのセットアップ (Colab シークレット方式)

OpenAI API の呼び出しには **API キー**による認証が必要です。
API キーは「あなたのアカウントで課金してよい」という証明書のようなものなので、
**コードに直接書いてはいけません**。Colab では **[シークレット]** 機能で安全に管理します。

**操作手順** (未登録の場合):
1. 画面左のサイドバーにある **鍵アイコン 🔑 [シークレット]** をクリック
2. **[新しいシークレットを追加]** を押す
3. 名前に `OPENAI_API_KEY`、値にあなたの API キーを入力
4. このノートブックからのアクセスを **オン** にする

次のセルは、Colab のシークレットからキーを読み取り、環境変数 `OPENAI_API_KEY` に設定します。
LangChain (`init_chat_model`) はこの環境変数を自動的に読むため、以降のコードにキーは一切登場しません。
Colab 以外の環境 (ローカル等) では、あらかじめ環境変数 `OPENAI_API_KEY` を設定しておけば動きます。


In [ ]:
import os

# Colab のシークレットから API キーを読み込み、環境変数に設定する
# Colab 以外の環境では except 側に入り、既存の環境変数 OPENAI_API_KEY をそのまま使う
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab シークレットから OPENAI_API_KEY を読み込みました。")
except ImportError:
    # Colab 以外では環境変数 OPENAI_API_KEY が設定済みとみなす
    print("Colab 以外の環境です。環境変数 OPENAI_API_KEY を使用します。")

# キーが読めているか (存在のみ) を確認。キー本体は表示しない
print("APIキー設定済み:", bool(os.environ.get("OPENAI_API_KEY")))

### 0-3. モデル名の準備

モデル名は変数 `MODEL` に集約します。`"openai:gpt-5.4"` という **`"provider:model"` 形式**の文字列です。
`:` の前 (`openai`) がプロバイダ名、後ろ (`gpt-5.4`) がモデル名を表します。

世代交代の速い領域なので、「モデル名を 1 箇所で管理していつでも差し替えられるようにする」こと自体が実務の定石です。


In [ ]:
# モデル名は変数に集約 ("provider:model" 形式)。研修実施時に最新へ差し替え
MODEL = "openai:gpt-5.4"

print("準備完了。使用モデル:", MODEL)

---

## 3-1. モデルの初期化 — init_chat_model

LangChain におけるすべての出発点が「チャットモデルの初期化」です。
第2章では `OpenAI()` クライアントを直接生成しましたが、LangChain では `init_chat_model` がその役割を担います。

import 元は `langchain.chat_models` です。`"openai:gpt-5.4"` のように文字列 1 つを渡すだけで初期化できます。
Anthropic に切り替えたければ `"anthropic:claude-sonnet-4-6"` と**文字列を書き換えるだけ**で済み、
後続のコードには一切手が入りません。これが「エンジンを載せ替えやすい設計」の核心です。

**ここで注目すること**: `invoke` の戻り値は**文字列ではなく `AIMessage` オブジェクト**です。
テキスト本文は `.text` で取り出せますが、オブジェクトの中には他にも多くの情報が詰まっています
(その解剖は 3-2 で行います)。


In [ ]:
from langchain.chat_models import init_chat_model

# "provider:model" 形式の文字列 1 つでモデルを初期化する
model = init_chat_model(MODEL)

response = model.invoke("LangChain とは何か、一言で説明して")

print("戻り値の型 :", type(response))   # <class 'langchain.messages.AIMessage'> — 文字列ではない!
print("本文(.text):", response.text)    # テキスト本文は .text で取り出す

### temperature で出力の揺れを観察する

第2章で学んだ API パラメータ (`temperature` など) は、`init_chat_model` のキーワード引数として
そのまま渡せます。`temperature` は出力のランダム性で、低いほど決定的・高いほど多様になります。

`temperature=0` と `temperature=1` で同じ質問を 3 回ずつ投げ、出力の揺れを比べてみましょう。

**期待される結果**: `temperature=0` では 3 回ともほぼ同じ句が、`temperature=1` では毎回違う句が返ります。
事実照会や分類のように**再現性が欲しいタスクでは低く**、発想系では高く、が使い分けの目安です。


In [ ]:
# temperature を 0 と 1 で各 3 回実行し、出力のばらつきを比較する
for temp in [0, 1]:
    print(f"--- temperature={temp} ---")
    # temperature はモデル初期化時のキーワード引数として渡す
    m = init_chat_model(MODEL, temperature=temp)
    for _ in range(3):
        r = m.invoke("AI をテーマに俳句を 1 句詠んでください。")
        print(r.text)
    print()

> **補足**: `init_chat_model` には LangChain 側で追加された `max_retries` というパラメータもあります。
> ネットワークエラーやレート制限 (429)、サーバーエラー (5xx) で呼び出しが失敗したとき、
> チャットモデルは**デフォルトで最大 6 回、指数バックオフ付きで自動リトライ**します
> (認証エラー 401 のようなリトライしても無駄なエラーはリトライしません)。
> 第2章では自分で書く必要があった「失敗したら投げ直す」処理も、フレームワークが肩代わりしてくれます。
>
> **補足 2**: reasoning 系モデルでは `temperature` が指定不可の場合があります。
> モデルを切り替えてこのセルでパラメータ関連のエラーが出たら、それは「このモデルは temperature 非対応」のサインです。


---

## 3-2. Messages — 会話を構成する標準データ型

前節で確認したとおり、モデルの応答は `AIMessage` オブジェクトでした。
実はモデルへの**入力**も、メッセージオブジェクトのリストで表現します。
第2章で `{"role": "user", "content": "..."}` という dict の配列を手で組み立てた、あの作業の型付き版です。

メッセージ型は 4 つあり、第2章の API ロールと **1 対 1 に対応**します。

| LangChain のメッセージ型 | 第2章の API ロール | 役割 |
|---|---|---|
| `SystemMessage` | `system` | モデルの振る舞い・人格・ガイドラインの指示 |
| `HumanMessage` | `user` | ユーザーからの入力 |
| `AIMessage` | `assistant` | モデルの応答 (テキスト・tool_calls を含む) |
| `ToolMessage` | `tool` | ツール実行結果のモデルへの返却 |

import 元は `langchain.messages` です。複数ターンの会話履歴を組み立てて、モデルに渡してみましょう。
3 つ目の `AIMessage` を**自分で書いて**履歴に挿入している点に注目してください
(「モデルがこう答えたことにする」一手として履歴に組み込めます。第2章と同じ原則です)。


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("あなたは英日翻訳のアシスタントです。"),
    HumanMessage("Translate: I love programming."),
    AIMessage("私はプログラミングが大好きです。"),   # 過去のモデル応答として履歴に挿入
    HumanMessage("Translate: I love building agents."),
]

response = model.invoke(messages)
print(response.text)   # => "私はエージェントを構築するのが大好きです。" など

### dict 形式も等価 — 第2章のコードがそのまま通用する

嬉しい事実を 1 つ。LangChain のモデルは、**第2章で使った OpenAI 形式の dict もそのまま受け付けます**。
下のセルは上のメッセージオブジェクト版と**等価**です (混在も可能)。

型付きオブジェクトを使う利点は、IDE の補完や型チェックが効くこと、そして次に見る豊富な属性に
素直にアクセスできることです。本書では原則メッセージオブジェクトを使い、
エージェントへの入力 (3-3) のような短い場面では dict 形式も使います。


In [ ]:
# OpenAI 形式の dict でも同じことができる (上のセルと等価)
messages_dict = [
    {"role": "system", "content": "あなたは英日翻訳のアシスタントです。"},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "私はプログラミングが大好きです。"},
    {"role": "user", "content": "Translate: I love building agents."},
]

response_dict = model.invoke(messages_dict)
print(response_dict.text)   # メッセージオブジェクト版と同じ結果が返る

### AIMessage の解剖 — usage_metadata と content_blocks

`AIMessage` には本文以外にも多くの情報が詰まっています。代表的な 2 つを観察しましょう。

- **`usage_metadata`** — トークン使用量。第2章の `usage` と同じ情報がプロバイダ共通の形式で入っています。
  `input_tokens` が第2章の `prompt_tokens`、`output_tokens` が `completion_tokens` に相当します。
- **`content_blocks`** — `content` (生のコンテンツ) をプロバイダ非依存の標準形式にパースしたものです。
  推論過程などプロバイダごとに形式が異なる応答を、`{"type": "text", ...}` のような**統一形式のブロックのリスト**に
  変換して返します。「入力の差異は `init_chat_model` が、出力の差異は `content_blocks` が吸収する」ことで、
  入口と出口の両方がプロバイダ非依存になります。

**期待される結果**: トークン数の dict と、`[{'type': 'text', 'text': '...'}]` 形式のブロックリストが表示されます。


In [ ]:
# AIMessage の付帯情報を観察する
response = model.invoke("こんにちは!一言であいさつを返してください。")

print("text          :", response.text)
print("usage_metadata:", response.usage_metadata)   # {'input_tokens': .., 'output_tokens': .., 'total_tokens': ..}
print("content_blocks:", response.content_blocks)   # [{'type': 'text', 'text': '...'}]


---

## 3-3. create_agent — 最初のエージェント

本章の山場です。推論エンジン (3-1) と、エンジンとやり取りするデータ形式 (3-2) が揃いました。
ここで両者を**ハーネス (Harness)** に組み込んでエージェントを起動します。

公式ドキュメントはエージェントを簡潔に定義しています——
「**エージェントとは、タスクが完了するまでループの中でツールを呼び続けるモデルである**」。
第2章の while ループの正体そのものです。そして、このループとそれを取り囲む一切合切
(モデル・プロンプト・ツール) をまとめて**ハーネス**と呼びます。**Agent = Model + Harness** です。

`create_agent` の設計思想は明快です。**案件固有の部分 (ツール・役割・入力) だけを引数で受け取り、
定型処理 (tool_calls の取り出し・`json.loads`・関数実行・`tool_call_id` を一致させた履歴の積み増し・
ループの終了判定) はすべてハーネスが引き受ける。** import 元は `langchain.agents` です。


In [ ]:
from langchain.agents import create_agent


# ツールとして使う関数。docstring と型ヒントからツールスキーマが自動生成される
# (第2章で手書きした JSON Schema が「消えた」のはこのため。仕組みの詳細は 3-4 で扱う)
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=MODEL,                                   # ① 推論エンジン (3-1 の知識)
    tools=[get_weather],                           # ② 使えるツールのリスト
    system_prompt="You are a helpful assistant",   # ③ 役割の指示 (3-2 の SystemMessage に相当)
)

# 入力は "messages" キーを持つ dict。エージェントは内部に State を持ち、入力はその更新差分として渡す
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)

# 最終応答 (messages の末尾) を content_blocks で取り出す
print(result["messages"][-1].content_blocks)

### 軌跡を読む — result["messages"] の 4 つのメッセージ

エージェント開発の最重要スキルの 1 つが、**実行軌跡を読み解く力**です。
`result["messages"]` には、エージェントの 1 回の実行で積み上がった全メッセージが入っています。
1 つずつダンプして、第2章の手動ループと見比べてみましょう。


In [ ]:
# 実行軌跡をダンプする。型・content・tool_calls を 1 メッセージずつ表示
for m in result["messages"]:
    print(type(m).__name__, "|", repr(m.content), "|", getattr(m, "tool_calls", None))

出力はおおむね次の 4 行になります (ツールを 1 回呼ぶ場合)。
これを ReAct ループの各ステップに対応付けて読んでください。

```text
HumanMessage  What's the weather in San Francisco?   None
AIMessage     (content は空)                          [{'name': 'get_weather', 'args': {'city': 'San Francisco'}, ...}]
ToolMessage   It's always sunny in San Francisco!    None
AIMessage     The weather in San Francisco is sunny! []
```

1. **`HumanMessage`** — 皆さんの入力。
2. **`AIMessage` (tool_calls 付き)** — モデルの 1 回目の応答。本文は空で、代わりに `tool_calls` に
   「get_weather を city='San Francisco' で呼べ」という宣言が入る。第2章のステップ② (`finish_reason="tool_calls"`) と同じ瞬間。
3. **`ToolMessage`** — ハーネスが `get_weather` を実行し、結果を積んだもの。`tool_call_id` は直前の宣言の `id` と一致。
   第2章のステップ③④で皆さんが手書きした部分を、ハーネスが肩代わりした証拠。
4. **`AIMessage` (最終応答)** — ツール結果を踏まえた 2 回目の応答。`tool_calls` は空になり、ループが終了。

### 第2章の手動ループとの対比 — 何が「消えた」のか

下の表が、本章の核心です。第2章の手動ループと diff を取ると——

| 第2章で**手書き**した処理 | 本章 (create_agent) では |
|---|---|
| ツールスキーマ (JSON Schema) の手書き | docstring + 型ヒントから**自動生成** |
| `tool_calls` の取り出しと `json.loads` | ハーネスが**自動実行** |
| アプリ側での関数実行 | ハーネスが**自動実行** |
| 宣言と結果のペアを正しい順序で履歴に積む (`tool_call_id` 一致) | ハーネスが**自動実行** (id を取り違える余地なし) |
| `finish_reason != "tool_calls"` の終了判定と while ループ制御 | ハーネスが**自動実行** |

**残ったのは「ツール関数の中身」「役割の指示」「入力」——案件固有の部分だけ**です。これがハーネスの価値です。
(第2章で完成させた手動ループ Notebook を開いて、上の軌跡と並べて見比べてみてください。)

なお、ツールと無関係の質問 (例: 「1+1 は?」) を投げると、モデルはツールを呼ばずに直接回答し、
軌跡は Human → AI の 2 メッセージだけになります。ツールを使うか判断しているのは
ハーネスではなく**モデル自身**である、という事実も軌跡から読み取れます。


In [ ]:
# ツール不要の質問では軌跡が Human → AI の 2 つだけになることを確認
result_no_tool = agent.invoke(
    {"messages": [{"role": "user", "content": "1+1 は?"}]}
)
for m in result_no_tool["messages"]:
    print(type(m).__name__, "|", repr(m.content), "|", getattr(m, "tool_calls", None))

### 戻り値は CompiledStateGraph

`create_agent` が返すものの正体も確認しておきましょう。


In [ ]:
# create_agent の戻り値の型を確認する
print(type(agent))
# <class 'langgraph.graph.state.CompiledStateGraph'>

`CompiledStateGraph`——LangGraph という名前空間のクラスです。
langchain パッケージは内部で langgraph (オーケストレーションランタイム) の上に構築されており、
`create_agent` は「ReAct ループを実行するグラフ」をコンパイルして返しています。
このグラフは `invoke` (一括実行) と `stream` (逐次実行) を持つ実行可能なオブジェクトです。
いまは「エージェントの実体は実行可能なグラフである」とだけ押さえてください
(グラフを自分で組む方法は第8章で扱います)。

> **古い API に注意**: エージェント構築を検索すると `initialize_agent` / `AgentExecutor` /
> `create_react_agent` を使う記事が大量に出てきますが、これらは **LangChain v0.x 時代の API であり、
> v1 では動きません** (旧 API は langchain-classic という別パッケージに分離されました)。
> 第2章の「`functions=` を見たら古い記事」と同じ判別法です。一次情報は常に現行ドキュメント (docs.langchain.com)。


### ステートレス性の体験 — このエージェントは「さっき」を知らない

重要な実験を 1 つ。同じ `agent` に対して、続けて 2 回 invoke してみるとどうなるでしょうか。

**期待される結果**: 2 回目の応答で、エージェントは「さっきの都市」が何を指すか分かっていません。
大阪の天気は答えても、東京との比較はできない——**直前の会話を覚えていない**のです。


In [ ]:
# 1 回目と 2 回目を別々に invoke する (同じ agent に対して)
r1 = agent.invoke({"messages": [{"role": "user", "content": "東京の天気は?"}]})
print("【1回目】", r1["messages"][-1].text)

r2 = agent.invoke({"messages": [{"role": "user", "content": "じゃあ大阪は? さっきの都市と比べてどう?"}]})
print("【2回目】", r2["messages"][-1].text)   # 「さっきの都市」が何か分からない

原因は第2章で学んだ原理に立ち返れば明らかです。API はステートレスであり、会話の継続は
アプリ側による全履歴の送信で実現されていました。`create_agent` のループは **1 回の invoke の中では**
履歴を自動で積みますが、**invoke をまたいだ記憶**はどこにも保存していません。
2 回目の invoke は、まっさらな State から始まっているのです。

会話を記憶するエージェントにする方法は、本章では解決しない**未解決の問題**として持ち越します。
答えは第4章で学ぶ **checkpointer** です。`create_agent` に引数を 1 つ足すだけで、この問題が解決します。


---

## 3-4. Tools — @tool によるツール定義と docstring の品質

前節では素の関数を `tools` に渡すだけでエージェントが動きました。しかし実務では、
**ツール定義の品質がエージェントの精度を直接左右します**。なぜなら——

LangChain におけるツールは「**スキーマ (名前・説明・引数定義) と、実行される関数のペア**」であり、
このうち**モデルに見えているのはスキーマだけ**だからです (関数本体のコードはモデルに送られません)。
つまり**ツールの docstring と引数名は、コメントではなくモデルへの指示文**なのです。

`@tool` デコレータ (`from langchain.tools import tool`) を使うと、関数からスキーマを生成する過程を
明示的に制御できます。スキーマの各部分が関数のどこから生成されるかを対応付けて理解してください。

- **名前** — 関数名がツール名になる (`snake_case` 推奨)
- **説明 (description)** — **docstring の本文**がそのまま description になる。モデルはこれを読んで「いつ使うか」を判断する
- **引数スキーマ** — **型ヒント**から生成される (このため型ヒントは必須)。docstring の `Args:` も引数説明として取り込まれる


In [ ]:
from langchain.tools import tool


@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"


# @tool でラップすると、生成されたスキーマ (name / description / 引数) を確認できる
print("name        :", search_database.name)          # => search_database (関数名)
print("description :", search_database.description)    # => docstring の本文
print("args        :", search_database.args)           # => 型ヒントから生成された引数スキーマ

### docstring の品質がツール選択精度を左右する — 比較実験

本節のコアとなる体験です。同じ「天気を取得する」機能でも、docstring が曖昧なツールと明確なツールでは、
モデルのツール選択の安定性が変わります。2 種類のツールを用意して、エージェントの振る舞いを比べてみましょう。

複数のツールを持つエージェントでは、モデルは description を**読み比べて**使うツールを選びます。
だから曖昧な説明は誤選択や不要な呼び出しの温床になります。

ここでは「天気を取得する `get_weather`」に加えて、ニュースを返す 2 つ目のツールを 2 パターン用意します。

- **パターン A (明確)**: docstring に「指定した都市の最新ニュースの見出しを取得する」と明確に書く
- **パターン B (曖昧)**: docstring が「データを取得する」としか書かれていない

まず、共通で使う 2 つのツールを定義します。


In [ ]:
@tool
def get_weather_tool(city: str) -> str:
    """指定した都市の現在の天気を取得する。天気・気温の問い合わせにはこのツールを使う。"""
    return f"{city}の天気: 晴れ、気温 24 度"


# --- パターン A: 明確な docstring のニュースツール ---
@tool
def get_news_clear(city: str) -> str:
    """指定した都市の最新ニュースの見出しを取得する。ニュース・話題・出来事の問い合わせにはこのツールを使う。"""
    return f"{city}の最新ニュース: 新しい公園がオープンしました"


# --- パターン B: 曖昧な docstring のニュースツール (中身は A と同じ) ---
@tool
def get_data_vague(city: str) -> str:
    """データを取得する。"""
    return f"{city}の最新ニュース: 新しい公園がオープンしました"


print("3 つのツールを定義しました。")

次に、それぞれのツールセットでエージェントを作り、**同じ質問**「東京のニュースを教えて」を投げます。
どちらのツールが呼ばれたか (= 軌跡の `AIMessage(tool_calls)`) を観察してください。

**期待される結果**: パターン A (明確) では `get_news_clear` が安定して選ばれます。
パターン B (曖昧) では「データを取得する」だけでは何のツールか伝わりにくく、選択が不安定になりがちです
(モデル・実行タイミングによっては天気ツールを誤って呼んだり、ツールを呼ばず直接回答することもあります)。
これが「docstring はモデルへの指示文」ということの実演です。


In [ ]:
def which_tool_called(result):
    """軌跡から、呼ばれたツール名のリストを取り出すヘルパー。"""
    names = []
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            names.append(tc["name"])
    return names


question = "東京のニュースを教えて"

# パターン A: 明確な docstring のニュースツールを含むエージェント
agent_clear = create_agent(model=MODEL, tools=[get_weather_tool, get_news_clear])
res_a = agent_clear.invoke({"messages": [{"role": "user", "content": question}]})
print("【A: 明確】呼ばれたツール:", which_tool_called(res_a))
print("【A: 明確】最終応答     :", res_a["messages"][-1].text)
print()

# パターン B: 曖昧な docstring のツールを含むエージェント
agent_vague = create_agent(model=MODEL, tools=[get_weather_tool, get_data_vague])
res_b = agent_vague.invoke({"messages": [{"role": "user", "content": question}]})
print("【B: 曖昧】呼ばれたツール:", which_tool_called(res_b))
print("【B: 曖昧】最終応答     :", res_b["messages"][-1].text)

よい description の指針は「**何をするツールか・いつ使うべきか**」を簡潔に書くことです。
これは新しいスキルではありません——チームメンバーに使ってもらう社内 API のドキュメントを書くのと同じ感覚です。
読み手が人間からモデルに変わっただけで、「読み手が迷わない説明を書く」という原則は共通です。

この「docstring の品質」の感覚は、次の演習 3-B で `search_faq` ツールの docstring を自分で書くときに
直接効いてきます。


---

## まとめ — ハンズオン 3-A で確認したこと

| 節 | 確認したこと | キーポイント |
|---|---|---|
| 3-1 | `init_chat_model` | `"provider:model"` 形式で初期化。戻り値は文字列ではなく `AIMessage`。`temperature` で出力の揺れ |
| 3-2 | Messages | 4 つの型が API ロールと 1 対 1 対応。dict 形式も等価。`usage_metadata` / `content_blocks` を観察 |
| 3-3 | create_agent | Agent = Model + Harness。軌跡 (Human → AI(tool_calls) → Tool → AI) を読む。戻り値は `CompiledStateGraph`。**ステートレス** |
| 3-4 | @tool | スキーマ + 関数のペア。モデルに見えるのはスキーマだけ。**docstring はモデルへの指示文**。品質が選択精度を左右 |

第2章で手書きしたループが `create_agent` の数行に置き換わり、消えたコード (スキーマ手書き・パース・履歴管理・
ループ制御) の行方を 1 つずつ突き止めました。

### 次は演習 3-B へ

このハンズオンで習得した部品を総動員して、演習 3-B では「**社内 IT ヘルプデスクエージェント v1**」を構築します
(ヘルプデスク Step 2)。第2章で手動実装したヘルプデスク QA ループを `create_agent` で書き直し、
FAQ 検索ツール (`search_faq`) を `@tool` で追加し、応答を `SupportAnswer` (Pydantic) で構造化します
(構造化出力 = 本章 3-5 の `response_format` は、演習で初めて手を動かして学びます)。
`# TODO` を埋める形式です。お楽しみに。
